In [66]:
import pandas as pd
from pathlib import Path

AISLES_PATH = Path("../data/raw/aisles.csv")
DEPARTMENTS_PATH = Path("../data/raw/departments.csv")
ORDER_PRODUCTS_PRIOR_PATH = Path("../data/raw/order_products__prior.csv")
ORDERS_PATH = Path("../data/raw/orders.csv")
PRODUCTS_PATH = Path("../data/raw/products.csv")

aisles_df = pd.read_csv(AISLES_PATH, dtype={
    "aisle_id": "int16"
})
departments_df = pd.read_csv(DEPARTMENTS_PATH, dtype={
    "department_id": "int8"
})
order_products_prior_df = pd.read_csv(ORDER_PRODUCTS_PRIOR_PATH, dtype={
    "order_id": "int32",
    "product_id": "int32",
    "add_to_cart_order": "int16",
    "reordered": "int8"
})
# data type optimization to reduce RAM 
orders_df = pd.read_csv(ORDERS_PATH, dtype={
    "order_id": "int32",
    "user_id": "int32",
    "order_number": "int16",
    "order_dow": "int8",
    "order_hour_of_day": "int8",
})
products_df = pd.read_csv(PRODUCTS_PATH, dtype={
    "product_id": "int32",
    "aisle_id": "int16",
    "department_id": "int8"
})

aisles_df.head()

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


In [67]:
aisles_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   aisle_id  134 non-null    int16
 1   aisle     134 non-null    str  
dtypes: int16(1), str(1)
memory usage: 3.4 KB


## Orders Dataframe

In [68]:
# look at the df
orders_df.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [69]:
# check shape
orders_df.shape

(3421083, 7)

In [70]:
# check data types
orders_df.dtypes

order_id                    int32
user_id                     int32
eval_set                      str
order_number                int16
order_dow                    int8
order_hour_of_day            int8
days_since_prior_order    float64
dtype: object

In [71]:
# check missing values
orders_df.isna().sum()

order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64

In [72]:
# check duplicates
orders_df.order_id.duplicated().sum()

np.int64(0)

## Order Products Prior Dataframe

In [73]:
# check dataframe
order_products_prior_df.head()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [74]:
# check info
order_products_prior_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 32434489 entries, 0 to 32434488
Data columns (total 4 columns):
 #   Column             Dtype
---  ------             -----
 0   order_id           int32
 1   product_id         int32
 2   add_to_cart_order  int16
 3   reordered          int8 
dtypes: int16(1), int32(2), int8(1)
memory usage: 340.3 MB


In [75]:
# check na values
order_products_prior_df.isna().sum()

order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64

In [76]:
orders_prior = orders_df[orders_df.eval_set == "prior"]

# merge orders and prior
master_df = orders_prior.merge(order_products_prior_df, how="inner", on="order_id")

In [77]:
# look at new dataframe
master_df.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered
0,2539329,1,prior,1,2,8,NaN,196,1,0
1,2539329,1,prior,1,2,8,NaN,14084,2,0
2,2539329,1,prior,1,2,8,NaN,12427,3,0
3,2539329,1,prior,1,2,8,NaN,26088,4,0
4,2539329,1,prior,1,2,8,NaN,26405,5,0


In [78]:
# check shape
master_df.shape

(32434489, 10)

In [79]:
# verify join
assert master_df.product_id.isna().sum() == 0

## Products Dataframe

In [80]:
# look at the dataframe
products_df.head()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [81]:
# info
products_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 49688 entries, 0 to 49687
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   product_id     49688 non-null  int32
 1   product_name   49688 non-null  str  
 2   aisle_id       49688 non-null  int16
 3   department_id  49688 non-null  int8 
dtypes: int16(1), int32(1), int8(1), str(1)
memory usage: 2.2 MB


In [82]:
# check na values
products_df.isna().sum()

product_id       0
product_name     0
aisle_id         0
department_id    0
dtype: int64

In [83]:
# merge df with products
master_df = master_df.merge(products_df, on="product_id", how="left")

In [84]:
# check new df
master_df.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id
0,2539329,1,prior,1,2,8,NaN,196,1,0,Soda,77,7
1,2539329,1,prior,1,2,8,NaN,14084,2,0,Organic Unsweetened Vanilla Almond Milk,91,16
2,2539329,1,prior,1,2,8,NaN,12427,3,0,Original Beef Jerky,23,19
3,2539329,1,prior,1,2,8,NaN,26088,4,0,Aged White Cheddar Popcorn,23,19
4,2539329,1,prior,1,2,8,NaN,26405,5,0,XL Pick-A-Size Paper Towel Rolls,54,17


In [85]:
# verify join
assert master_df.product_name.isna().sum() == 0

## Aisles Dataframe

In [86]:
aisles_df.head()

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


In [87]:
# check info
aisles_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   aisle_id  134 non-null    int16
 1   aisle     134 non-null    str  
dtypes: int16(1), str(1)
memory usage: 3.4 KB


In [88]:
# shape
aisles_df.shape

(134, 2)

In [89]:
# na values
aisles_df.isna().sum()

aisle_id    0
aisle       0
dtype: int64

In [90]:
# merge df and aisles
master_df = master_df.merge(aisles_df, on="aisle_id", how="left")

In [91]:
# check new df
master_df.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle
0,2539329,1,prior,1,2,8,NaN,196,1,0,Soda,77,7,soft drinks
1,2539329,1,prior,1,2,8,NaN,14084,2,0,Organic Unsweetened Vanilla Almond Milk,91,16,soy lactosefree
2,2539329,1,prior,1,2,8,NaN,12427,3,0,Original Beef Jerky,23,19,popcorn jerky
3,2539329,1,prior,1,2,8,NaN,26088,4,0,Aged White Cheddar Popcorn,23,19,popcorn jerky
4,2539329,1,prior,1,2,8,NaN,26405,5,0,XL Pick-A-Size Paper Towel Rolls,54,17,paper goods


In [92]:
# verify join
assert master_df.aisle.isna().sum() == 0

In [93]:
# check shape
master_df.shape

(32434489, 14)

In [94]:
# check na values
master_df.isna().sum()

order_id                        0
user_id                         0
eval_set                        0
order_number                    0
order_dow                       0
order_hour_of_day               0
days_since_prior_order    2078068
product_id                      0
add_to_cart_order               0
reordered                       0
product_name                    0
aisle_id                        0
department_id                   0
aisle                           0
dtype: int64

## Departments Dataframe

In [95]:
# check it
departments_df.head()

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


In [96]:
# info about it
departments_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   department_id  21 non-null     int8 
 1   department     21 non-null     str  
dtypes: int8(1), str(1)
memory usage: 491.0 bytes


In [97]:
# check na values
departments_df.isna().sum()

department_id    0
department       0
dtype: int64

In [98]:
# merge df and departments
master_df = master_df.merge(departments_df, on="department_id", how="left")

In [99]:
# check final dataframe
master_df.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle,department
0,2539329,1,prior,1,2,8,NaN,196,1,0,Soda,77,7,soft drinks,beverages
1,2539329,1,prior,1,2,8,NaN,14084,2,0,Organic Unsweetened Vanilla Almond Milk,91,16,soy lactosefree,dairy eggs
2,2539329,1,prior,1,2,8,NaN,12427,3,0,Original Beef Jerky,23,19,popcorn jerky,snacks
3,2539329,1,prior,1,2,8,NaN,26088,4,0,Aged White Cheddar Popcorn,23,19,popcorn jerky,snacks
4,2539329,1,prior,1,2,8,NaN,26405,5,0,XL Pick-A-Size Paper Towel Rolls,54,17,paper goods,household


In [100]:
# verify join
assert master_df.department.isna().sum() == 0

In [101]:
# check info
master_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 32434489 entries, 0 to 32434488
Data columns (total 15 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int32  
 1   user_id                 int32  
 2   eval_set                str    
 3   order_number            int16  
 4   order_dow               int8   
 5   order_hour_of_day       int8   
 6   days_since_prior_order  float64
 7   product_id              int32  
 8   add_to_cart_order       int16  
 9   reordered               int8   
 10  product_name            str    
 11  aisle_id                int16  
 12  department_id           int8   
 13  aisle                   str    
 14  department              str    
dtypes: float64(1), int16(3), int32(3), int8(4), str(4)
memory usage: 3.5 GB


In [102]:
# check na values
master_df.isna().sum()

order_id                        0
user_id                         0
eval_set                        0
order_number                    0
order_dow                       0
order_hour_of_day               0
days_since_prior_order    2078068
product_id                      0
add_to_cart_order               0
reordered                       0
product_name                    0
aisle_id                        0
department_id                   0
aisle                           0
department                      0
dtype: int64

In [103]:
# check shape
master_df.shape

(32434489, 15)

In [104]:
# convert aisle and department columns to category so that memory is reduced
master_df.aisle = master_df.aisle.astype("category")
master_df.department = master_df.department.astype("category")

# also eval_set
master_df.eval_set = master_df.eval_set.astype("category")

## Creating aggregate data

Users

In [105]:
users_agg = (master_df.groupby("user_id").agg({
    "order_id": "nunique",
    "product_id": "count",
    "reordered": "mean"
    }).reset_index()
)
# rename
users_agg.rename(columns={
    "order_id": "unique_orders",
    "product_id": "total_products_purchased",
    "reordered": "reordered_average"
}, inplace=True)

In [106]:
# see the new users dataframe
users_agg.head()

,user_id,unique_orders,total_products_purchased,reordered_average
0,1,10,59,0.694915
1,2,14,195,0.476923
2,3,12,88,0.625000
3,4,5,18,0.055556
4,5,4,37,0.378378


Products

In [107]:
products_agg = (master_df.groupby("product_id").agg({
        "order_id": "nunique",
        "user_id": "nunique",
        "reordered": "mean",
    }).reset_index()
)

# rename columns
products_agg.rename(columns={
    "order_id": "unique_order_ids",
    "user_id": "unique_user_ids",
    "reordered": "reordered_average"
}, inplace=True)

In [108]:
# see the products dataframe
products_agg.head()

,product_id,unique_order_ids,unique_user_ids,reordered_average
0,1,1852,716,0.613391
1,2,90,78,0.133333
2,3,277,74,0.732852
3,4,329,182,0.446809
4,5,15,6,0.600000


Basket Sizes

In [109]:
basket_sizes = (master_df.groupby("order_id").size().reset_index(name="basket_size"))

# merge with users
user_basket_features = basket_sizes.merge(master_df[["order_id", "user_id"]].drop_duplicates(), on="order_id")

# aggregate
avg_basket_sizes = pd.DataFrame((user_basket_features.groupby("user_id")["basket_size"].mean()))

In [110]:
# see the dataframe
avg_basket_sizes.head()

,basket_size
user_id,
1,5.900000
2,13.928571
3,7.333333
4,3.600000
5,9.250000


## Saving using parquet (faster)

In [ ]:
# save users df
users_agg.to_parquet("../data/processed/users_df.parquet", index=False)
# save products df
products_agg.to_parquet("../data/processed/products_df.parquet", index=False)
# save avg. basket sizes df
avg_basket_sizes.to_parquet("../data/processed/avg_basket_sizes_df.parquet", index=False)
#save master df
master_df.to_parquet("../data/processed/master_df_cleaned.parquet", index=False)